In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import pandas as pd
from joblib import load
from sklearn.model_selection import train_test_split
import numpy as np


In [2]:
#nacteni pipeline
preprocessor = load("preprocessor.joblib")
#nacteni dat
df = pd.read_csv("netflix_customer_churn.csv")
#normalizace dat
# Odstranění duplicit
df = df.drop_duplicates()

# Normalizace textových hodnot
text_cols = [
    "gender",
    "subscription_type",
    "region",
    "device",
    "payment_method",
    "favorite_genre"
]

for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

# Logicky nesmyslné hodnoty → NaN
df.loc[(df["age"] < 0) | (df["age"] > 120), "age"] = np.nan

df.loc[df["watch_hours"] < 0, "watch_hours"] = np.nan
df.loc[df["avg_watch_time_per_day"] < 0, "avg_watch_time_per_day"] = np.nan

df.loc[df["last_login_days"] < 0, "last_login_days"] = np.nan

df.loc[df["number_of_profiles"] < 0, "number_of_profiles"] = np.nan

df.loc[df["monthly_fee"] < 0, "monthly_fee"] = np.nan

df.isna().sum()
#split dat
X = df.drop(columns=["churned", "customer_id"])
y = df["churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
#aplikace preprocessing pipeline
X_train_prepared = preprocessor.transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

Logisticá Regrese:

In [3]:
#logistická regrese
log_reg = LogisticRegression(
    max_iter=1000,
    random_state=42
)
#trénování na datech
log_reg.fit(X_train_prepared, y_train)

#predikce
y_pred = log_reg.predict(X_test_prepared)

# predikce pravděpodobností
y_pred_proba = log_reg.predict_proba(X_test_prepared)[:, 1]


Počet iterací byl zvýšen, aby byla zajištěna konvergence logistické regrese při větším počtu vstupních proměnných.

Výpočet metrik

In [4]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC AUC  : {roc_auc:.4f}")

Accuracy : 0.8910
Precision: 0.8760
Recall   : 0.9125
F1-score : 0.8939
ROC AUC  : 0.9686


_____________________________________________________________________________________________________________________________________________

Rozhodovací strom

In [5]:
#inicializace stromu
dt = DecisionTreeClassifier(    
    random_state=42
)

dt.fit(X_train_prepared,y_train)
#predikce tříd
y_pred_dt = dt.predict(X_test_prepared)
y_pred_proba_dt = dt.predict_proba(X_test_prepared)[:, 1]

Výpočet metrik:

In [6]:
accuracy_dt = accuracy_score(y_test, y_pred_dt)
precision_dt = precision_score(y_test, y_pred_dt)
recall_dt = recall_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)
roc_auc_dt = roc_auc_score(y_test, y_pred_proba_dt)

print(f"Accuracy : {accuracy_dt:.4f}")
print(f"Precision: {precision_dt:.4f}")
print(f"Recall   : {recall_dt:.4f}")
print(f"F1-score : {f1_dt:.4f}")
print(f"ROC AUC  : {roc_auc_dt:.4f}")

print(f"Hloubka : {dt.get_depth():.0f}")
print(f"Listy : {dt.get_n_leaves():.0f}")

Accuracy : 0.9790
Precision: 0.9782
Recall   : 0.9801
F1-score : 0.9791
ROC AUC  : 0.9790
Hloubka : 10
Listy : 96


Kontrola přeučení rozhodovacího stromu:

In [7]:
# Výkon na TRAIN datech
y_train_proba = dt.predict_proba(X_train_prepared)[:, 1]
roc_train = roc_auc_score(y_train, y_train_proba)

# Výkon v CROSS-VALIDATION (cv=5)
cv_scores = cross_val_score(
    dt,
    X_train_prepared,
    y_train,
    cv=5,
    scoring="roc_auc"
)

roc_cv_mean = cv_scores.mean()
roc_cv_std = cv_scores.std()

# Výkon na TEST datech
y_test_proba = dt.predict_proba(X_test_prepared)[:, 1]
roc_test = roc_auc_score(y_test, y_test_proba)

print(f"TRAIN ROC AUC : {roc_train:.4f}")
print(f"CV ROC AUC    : {roc_cv_mean:.4f} ± {roc_cv_std:.4f}")
print(f"TEST ROC AUC  : {roc_test:.4f}")

TRAIN ROC AUC : 1.0000
CV ROC AUC    : 0.9773 ± 0.0030
TEST ROC AUC  : 0.9790


Rozdíl přibližně 0.021 mezi tréninkovým a validačním/testovacím ROC AUC je výrazný a indikuje přeučení modelu, protože výkon na trénovacích datech je podstatně vyšší než na datech neviděných. Model si tedy do značné míry pamatuje trénovací data, ale jeho schopnost generalizace je omezená. Z toho důvodu bude provedeno ladění Hyperparametrů v pozdější části práce

_____________________________________________________________________________________________________________________________________________

Random Forest:

In [8]:
# základní ranfom forest
rf = RandomForestClassifier(
    n_estimators=100,    
    random_state=42,    
    n_jobs=-1
)

zvoleno 100 stromů, jako vhodný kompromis mezi výkonností a časem

In [9]:
rf.fit(X_train_prepared,y_train)
# predikce tříd
y_pred_rf = rf.predict(X_test_prepared)
y_pred_proba_rf = rf.predict_proba(X_test_prepared)[:, 1]

vyhodnocení metrik:

In [10]:
accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
roc_auc_rf = roc_auc_score(y_test, y_pred_proba_rf)

print(f"Accuracy : {accuracy_rf:.4f}")
print(f"Precision: {precision_rf:.4f}")
print(f"Recall   : {recall_rf:.4f}")
print(f"F1-score : {f1_rf:.4f}")
print(f"ROC AUC  : {roc_auc_rf:.4f}")

depths = [tree.get_depth() for tree in rf.estimators_]
leaves = [tree.get_n_leaves() for tree in rf.estimators_]

print(f"Průměrná hloubka stromů: {np.mean(depths):.1f}")
print(f"Maximální hloubka stromů: {np.max(depths)}")

print(f"Průměrný počet listů: {np.mean(leaves):.1f}")
print(f"Maximální počet listů: {np.max(leaves)}")

Accuracy : 0.9760
Precision: 0.9743
Recall   : 0.9781
F1-score : 0.9762
ROC AUC  : 0.9975
Průměrná hloubka stromů: 18.4
Maximální hloubka stromů: 22
Průměrný počet listů: 369.9
Maximální počet listů: 571


Kontrola přeučení Random Forest:

In [11]:
# Výkon na TRAIN datech
y_train_proba = rf.predict_proba(X_train_prepared)[:, 1]
roc_train = roc_auc_score(y_train, y_train_proba)

# Výkon v CROSS-VALIDATION (cv=5)
cv_scores = cross_val_score(
    rf,
    X_train_prepared,
    y_train,
    cv=5,
    scoring="roc_auc"
)

roc_cv_mean = cv_scores.mean()
roc_cv_std = cv_scores.std()

# Výkon na TEST datech
y_test_proba = rf.predict_proba(X_test_prepared)[:, 1]
roc_test = roc_auc_score(y_test, y_test_proba)

print(f"TRAIN ROC AUC : {roc_train:.4f}")
print(f"CV ROC AUC    : {roc_cv_mean:.4f} ± {roc_cv_std:.4f}")
print(f"TEST ROC AUC  : {roc_test:.4f}")

TRAIN ROC AUC : 1.0000
CV ROC AUC    : 0.9967 ± 0.0015
TEST ROC AUC  : 0.9975


Random Forest dosahuje perfektního výkonu na trénovacích datech (ROC AUC = 1.0000) a jeho výkon v křížové validaci (0.9967 ± 0.0015) i na testovacích datech (0.9975) zůstává velmi vysoký a konzistentní. Malý rozdíl mezi tréninkovým, validačním a testovacím výkonem naznačuje, že model nevykazuje významné známky přeučení a velmi dobře generalizuje. Z tohoto důvodu není další ladění hyperparametrů nutné z hlediska zlepšení predikčního výkonu; případná optimalizace by se mohla zaměřit spíše na systémovou efektivitu modelu, například snížení výpočetní náročnosti nebo paměťových požadavků.

_____________________________________________________________________________________________________________________________________________

Gradient boosting:


In [12]:
gb = GradientBoostingClassifier(
    n_estimators=100,    
    learning_rate=0.1,  
    random_state=42
)

gb.fit(X_train_prepared,y_train)

# predikce tříd
y_pred_gb = gb.predict(X_test_prepared)
y_pred_proba_gb = gb.predict_proba(X_test_prepared)[:, 1]

In [13]:
accuracy_gb = accuracy_score(y_test, y_pred_gb)
precision_gb = precision_score(y_test, y_pred_gb)
recall_gb = recall_score(y_test, y_pred_gb)
f1_gb = f1_score(y_test, y_pred_gb)
roc_auc_gb = roc_auc_score(y_test, y_pred_proba_gb)

print(f"Accuracy : {accuracy_gb:.4f}")
print(f"Precision: {precision_gb:.4f}")
print(f"Recall   : {recall_gb:.4f}")
print(f"F1-score : {f1_gb:.4f}")
print(f"ROC AUC  : {roc_auc_gb:.4f}")

depths_gb = [tree[0].get_depth() for tree in gb.estimators_]
leaves_gb = [tree[0].get_n_leaves() for tree in gb.estimators_]

print(f"Průměrná hloubka stromů: {np.mean(depths_gb):.1f}")
print(f"Maximální hloubka stromů: {np.max(depths_gb)}")

print(f"Průměrný počet listů: {np.mean(leaves_gb):.1f}")
print(f"Maximální počet listů: {np.max(leaves_gb)}")

Accuracy : 0.9880
Precision: 0.9900
Recall   : 0.9861
F1-score : 0.9880
ROC AUC  : 0.9981
Průměrná hloubka stromů: 3.0
Maximální hloubka stromů: 3
Průměrný počet listů: 8.0
Maximální počet listů: 8


Kontrola přeučení Gradient Boosting:

In [14]:
# Výkon na TRAIN datech
y_train_proba = gb.predict_proba(X_train_prepared)[:, 1]
roc_train = roc_auc_score(y_train, y_train_proba)

# Výkon v CROSS-VALIDATION (cv=5)
cv_scores = cross_val_score(
    gb,
    X_train_prepared,
    y_train,
    cv=5,
    scoring="roc_auc"
)

roc_cv_mean = cv_scores.mean()
roc_cv_std = cv_scores.std()

# Výkon na TEST datech
y_test_proba = gb.predict_proba(X_test_prepared)[:, 1]
roc_test = roc_auc_score(y_test, y_test_proba)

print(f"TRAIN ROC AUC : {roc_train:.4f}")
print(f"CV ROC AUC    : {roc_cv_mean:.4f} ± {roc_cv_std:.4f}")
print(f"TEST ROC AUC  : {roc_test:.4f}")

TRAIN ROC AUC : 0.9998
CV ROC AUC    : 0.9988 ± 0.0012
TEST ROC AUC  : 0.9981


Gradient Boosting dosahuje velmi vysokého výkonu na trénovacích datech (ROC AUC = 0.9998), přičemž jeho výkon v křížové validaci (0.9988 ± 0.0012) i na testovacích datech (0.9981) zůstává konzistentní. Minimální rozdíly mezi tréninkovým, validačním a testovacím výkonem naznačují, že model nevykazuje významné známky přeučení a velmi dobře generalizuje. Z tohoto důvodu není další ladění hyperparametrů nutné z hlediska zlepšení predikčního výkonu a případná optimalizace by se mohla zaměřit spíše na snížení modelové složitosti nebo výpočetní náročnosti.

_____________________________________________________________________________________________________________________________________________

Ladění rozhodovacího stromu:

In [15]:
dt = DecisionTreeClassifier(    
    random_state=42
    )

param_grid = {
    "max_depth": [None, 5, 10, 15, 20],
    "min_samples_leaf": [1, 5, 10, 20],
    "min_samples_split": [2, 5, 10, 20]
}
grid_search_dt = GridSearchCV(
    estimator=dt,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1
)

#fitnutí stromu
grid_search_dt.fit(X_train_prepared, y_train)
#nejlepší hyperparametry
grid_search_dt.best_params_
#výkon z křížové validace
grid_search_dt.best_score_
#finální strom
best_dt = grid_search_dt.best_estimator_

výsledky:

In [16]:
y_pred_dt_tuned = best_dt.predict(X_test_prepared)
y_pred_proba_dt_tuned = best_dt.predict_proba(X_test_prepared)[:, 1]

accuracy_dt_tuned = accuracy_score(y_test, y_pred_dt_tuned)
precision_dt_tuned = precision_score(y_test, y_pred_dt_tuned)
recall_dt_tuned = recall_score(y_test, y_pred_dt_tuned)
f1_dt_tuned = f1_score(y_test, y_pred_dt_tuned)
roc_auc_dt_tuned = roc_auc_score(y_test, y_pred_proba_dt_tuned)

print(f"Accuracy : {accuracy_dt_tuned:.4f}")
print(f"Precision: {precision_dt_tuned:.4f}")
print(f"Recall   : {recall_dt_tuned:.4f}")
print(f"F1-score : {f1_dt_tuned:.4f}")
print(f"ROC AUC  : {roc_auc_dt_tuned:.4f}")

print(f"Hloubka : {best_dt.get_depth():.0f}")
print(f"Listy : {best_dt.get_n_leaves():.0f}")

Accuracy : 0.9790
Precision: 0.9859
Recall   : 0.9722
F1-score : 0.9790
ROC AUC  : 0.9946
Hloubka : 9
Listy : 59


In [17]:
# Výkon na TRAIN datech
y_train_proba = best_dt.predict_proba(X_train_prepared)[:, 1]
roc_train = roc_auc_score(y_train, y_train_proba)

# Výkon v CROSS-VALIDATION (cv=5)
cv_scores = cross_val_score(
    best_dt,
    X_train_prepared,
    y_train,
    cv=5,
    scoring="roc_auc"
)

roc_cv_mean = cv_scores.mean()
roc_cv_std = cv_scores.std()

# Výkon na TEST datech
y_test_proba = best_dt.predict_proba(X_test_prepared)[:, 1]
roc_test = roc_auc_score(y_test, y_test_proba)

print(f"TRAIN ROC AUC : {roc_train:.4f}")
print(f"CV ROC AUC    : {roc_cv_mean:.4f} ± {roc_cv_std:.4f}")
print(f"TEST ROC AUC  : {roc_test:.4f}")

TRAIN ROC AUC : 0.9995
CV ROC AUC    : 0.9916 ± 0.0028
TEST ROC AUC  : 0.9946


Z výsledků je patrné, že model nevykazuje výrazné rozdíly mezi výkonem v pětinásobné křížové validaci, na trénovacích datech a na testovací sadě. Tato konzistence výkonu představuje důkaz, že model není přeučený a vykazuje dobrou schopnost generalizace.

In [18]:
results_dt = pd.DataFrame(grid_search_dt.cv_results_)

params_expanded = results_dt["params"].apply(pd.Series)

summary_dt = pd.concat(
    [
        params_expanded,
        results_dt[["mean_test_score", "std_test_score", "rank_test_score"]]
    ],
    axis=1
).sort_values("rank_test_score")

summary_dt.head()


,max_depth,min_samples_leaf,min_samples_split,mean_test_score,std_test_score,rank_test_score
39,10.0,5.0,20.0,0.991607,0.002808,1
7,NaN,5.0,20.0,0.991607,0.002808,1
71,20.0,5.0,20.0,0.991607,0.002808,1
55,15.0,5.0,20.0,0.991607,0.002808,1
56,15.0,10.0,2.0,0.991264,0.002990,5


Na základě výsledků 5‑fold křížové validace byla jako nejlepší identifikována kombinace max_depth = 10, min_samples_leaf = 5 a min_samples_split = 20, která dosahuje nejvyšší průměrné ROC hodnoty napříč jednotlivými foldy.
Vyladěný strom má ROC=0.9946, oproti původnímu 0.9790

_____________________________________________________________________________________________________________________________________________

Ladění Random forest:

In [19]:
rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)
param_grid = {
    "n_estimators": [50, 100, 500],
    "max_depth": [None, 8, 12, 16],
    "min_samples_leaf": [1, 5, 10],
    "min_samples_split": [2, 10, 20],
    "max_features": ["sqrt", "log2"]
}

grid_search_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1
)

#fitnutí Random Forrest
grid_search_rf.fit(X_train_prepared, y_train)
#nejlepší hyperparametry
grid_search_rf.best_params_
#výkon z křížové validace
grid_search_rf.best_score_
#finální les
best_rf = grid_search_rf.best_estimator_


In [20]:
# predikce
y_pred_rf_tuned = best_rf.predict(X_test_prepared)
y_pred_proba_rf_tuned = best_rf.predict_proba(X_test_prepared)[:, 1]

# metriky
accuracy_rf_tuned = accuracy_score(y_test, y_pred_rf_tuned)
precision_rf_tuned = precision_score(y_test, y_pred_rf_tuned)
recall_rf_tuned = recall_score(y_test, y_pred_rf_tuned)
f1_rf_tuned = f1_score(y_test, y_pred_rf_tuned)
roc_auc_rf_tuned = roc_auc_score(y_test, y_pred_proba_rf_tuned)

depths_tuned = [tree.get_depth() for tree in best_rf.estimators_]
leaves_tuned = [tree.get_n_leaves() for tree in best_rf.estimators_]

print(f"Accuracy : {accuracy_rf_tuned:.4f}")
print(f"Precision: {precision_rf_tuned:.4f}")
print(f"Recall   : {recall_rf_tuned:.4f}")
print(f"F1-score : {f1_rf_tuned:.4f}")
print(f"ROC AUC  : {roc_auc_rf_tuned:.4f}")

# doplňkové info o modelu
print(f"n_estimators: {best_rf.n_estimators}")
print(f"Průměrná hloubka stromů: {np.mean(depths_tuned):.1f}")
print(f"Maximální hloubka stromů: {np.max(depths_tuned)}")

print(f"Průměrný počet listů: {np.mean(leaves_tuned):.1f}")
print(f"Maximální počet listů: {np.max(leaves_tuned)}")

Accuracy : 0.9810
Precision: 0.9821
Recall   : 0.9801
F1-score : 0.9811
ROC AUC  : 0.9980
n_estimators: 500
Průměrná hloubka stromů: 16.0
Maximální hloubka stromů: 16
Průměrný počet listů: 361.3
Maximální počet listů: 608


Testování přeučení:

In [21]:
# Výkon na TRAIN datech
y_train_proba = best_rf.predict_proba(X_train_prepared)[:, 1]
roc_train = roc_auc_score(y_train, y_train_proba)

# Výkon v CROSS-VALIDATION (cv=5)
cv_scores = cross_val_score(
    best_rf,
    X_train_prepared,
    y_train,
    cv=5,
    scoring="roc_auc"
)

roc_cv_mean = cv_scores.mean()
roc_cv_std = cv_scores.std()

# Výkon na TEST datech
y_test_proba = best_rf.predict_proba(X_test_prepared)[:, 1]
roc_test = roc_auc_score(y_test, y_test_proba)

print(f"TRAIN ROC AUC : {roc_train:.4f}")
print(f"CV ROC AUC    : {roc_cv_mean:.4f} ± {roc_cv_std:.4f}")
print(f"TEST ROC AUC  : {roc_test:.4f}")

TRAIN ROC AUC : 1.0000
CV ROC AUC    : 0.9972 ± 0.0015
TEST ROC AUC  : 0.9980


Random Forest dosahuje perfektního výkonu na trénovacích datech (ROC AUC = 1.0000), přičemž jeho výkon v pětinásobné křížové validaci (0.9972 ± 0.0015) i na testovacích datech (0.9980) zůstává velmi vysoký a konzistentní. Minimální rozdíly mezi těmito hodnotami naznačují, že model nevykazuje známky přeučení a velmi dobře generalizuje.

In [22]:
results_rf = pd.DataFrame(grid_search_rf.cv_results_)

params_expanded_rf = results_rf["params"].apply(pd.Series)

summary_rf = pd.concat(
    [
        params_expanded_rf,
        results_rf[["mean_test_score", "std_test_score", "rank_test_score"]]
    ],
    axis=1
).sort_values("rank_test_score")

summary_rf.head(
15)

,max_depth,max_features,min_samples_leaf,min_samples_split,n_estimators,mean_test_score,std_test_score,rank_test_score
164,16.0,sqrt,1,2,500,0.997186,0.001498,1
191,16.0,log2,1,2,500,0.997186,0.001498,1
2,NaN,sqrt,1,2,500,0.997080,0.001654,3
29,NaN,log2,1,2,500,0.997080,0.001654,3
110,12.0,sqrt,1,2,500,0.997046,0.001479,5
137,12.0,log2,1,2,500,0.997046,0.001479,5
190,16.0,log2,1,2,100,0.996882,0.001457,7
163,16.0,sqrt,1,2,100,0.996882,0.001457,8
136,12.0,log2,1,2,100,0.996761,0.001542,9
109,12.0,sqrt,1,2,100,0.996761,0.001542,9


Pro Random Forest byla zvolena konfigurace s parametry max_depth = 16, max_features = log2, min_samples_leaf = 1, min_samples_split = 2 a n_estimators = 200, která dosahuje velmi vysokého a stabilního výkonu v křížové validaci (ROC AUC = 0.997047 ± 0.001474). Počet stromů byl nastaven na 200, jelikož poskytuje srovnatelný predikční výkon jako vyšší hodnoty při nižší výpočetní náročnosti a lepší systémové efektivitě.

_____________________________________________________________________________________________________________________________________________

Ladění Gradient Boostingu:

In [23]:
gb = GradientBoostingClassifier(
    random_state=42
)
param_grid = {
    "n_estimators": [100, 200, 300],    
    "learning_rate": [0.05, 0.1, 0.2],    
    "max_depth": [2, 3, 4, 10],    
    "min_samples_leaf": [1, 5, 10, 20],    
    "subsample": [0.8, 1.0]
}

grid_search_gb = GridSearchCV(
    estimator=gb,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1
)

#fitnutí Random Forrest
grid_search_gb.fit(X_train_prepared, y_train)
#nejlepší hyperparametry
grid_search_gb.best_params_
#výkon z křížové validace
grid_search_gb.best_score_
#finální les
best_gb = grid_search_gb.best_estimator_


In [24]:
# predikce
y_pred_gb_tuned = best_gb.predict(X_test_prepared)
y_pred_proba_gb_tuned = best_gb.predict_proba(X_test_prepared)[:, 1]

# metriky
accuracy_gb_tuned = accuracy_score(y_test, y_pred_gb_tuned)
precision_gb_tuned = precision_score(y_test, y_pred_gb_tuned)
recall_gb_tuned = recall_score(y_test, y_pred_gb_tuned)
f1_gb_tuned = f1_score(y_test, y_pred_gb_tuned)
roc_auc_gb_tuned = roc_auc_score(y_test, y_pred_proba_gb_tuned)

depths_tuned = [tree[0].get_depth() for tree in best_gb.estimators_]
leaves_tuned = [tree[0].get_n_leaves() for tree in best_gb.estimators_]

print(f"Accuracy : {accuracy_gb_tuned:.4f}")
print(f"Precision: {precision_gb_tuned:.4f}")
print(f"Recall   : {recall_gb_tuned:.4f}")
print(f"F1-score : {f1_gb_tuned:.4f}")
print(f"ROC AUC  : {roc_auc_gb_tuned:.4f}")

# doplňkové info o modelu
print(f"n_estimators: {best_gb.n_estimators}")
print(f"Průměrná hloubka stromů: {np.mean(depths_tuned):.1f}")
print(f"Maximální hloubka stromů: {np.max(depths_tuned)}")

print(f"Průměrný počet listů: {np.mean(leaves_tuned):.1f}")
print(f"Maximální počet listů: {np.max(leaves_tuned)}")

Accuracy : 0.9950
Precision: 0.9980
Recall   : 0.9920
F1-score : 0.9950
ROC AUC  : 0.9990
n_estimators: 300
Průměrná hloubka stromů: 2.0
Maximální hloubka stromů: 2
Průměrný počet listů: 4.0
Maximální počet listů: 4


In [25]:
# Výkon na TRAIN datech
y_train_proba = best_gb.predict_proba(X_train_prepared)[:, 1]
roc_train = roc_auc_score(y_train, y_train_proba)

# Výkon v CROSS-VALIDATION (cv=5)
cv_scores = cross_val_score(
    best_gb,
    X_train_prepared,
    y_train,
    cv=5,
    scoring="roc_auc"
)

roc_cv_mean = cv_scores.mean()
roc_cv_std = cv_scores.std()

# Výkon na TEST datech
y_test_proba = best_gb.predict_proba(X_test_prepared)[:, 1]
roc_test = roc_auc_score(y_test, y_test_proba)

print(f"TRAIN ROC AUC : {roc_train:.4f}")
print(f"CV ROC AUC    : {roc_cv_mean:.4f} ± {roc_cv_std:.4f}")
print(f"TEST ROC AUC  : {roc_test:.4f}")

TRAIN ROC AUC : 1.0000
CV ROC AUC    : 0.9994 ± 0.0010
TEST ROC AUC  : 0.9990


In [26]:
results_gb = pd.DataFrame(grid_search_gb.cv_results_)

params_expanded_gb = results_gb["params"].apply(pd.Series)

summary_gb = pd.concat(
    [
        params_expanded_gb,
        results_gb[["mean_test_score", "std_test_score", "rank_test_score"]]
    ],
    axis=1
).sort_values("rank_test_score")

summary_gb.head(
15)

,learning_rate,max_depth,min_samples_leaf,n_estimators,subsample,mean_test_score,std_test_score,rank_test_score
202,0.2,2.0,5.0,300.0,0.8,0.999409,0.000985,1
196,0.2,2.0,1.0,300.0,0.8,0.999377,0.001051,2
197,0.2,2.0,1.0,300.0,1.0,0.999361,0.001129,3
209,0.2,2.0,10.0,300.0,1.0,0.999341,0.001160,4
214,0.2,2.0,20.0,300.0,0.8,0.999336,0.001136,5
208,0.2,2.0,10.0,300.0,0.8,0.999334,0.001073,6
215,0.2,2.0,20.0,300.0,1.0,0.999331,0.001165,7
203,0.2,2.0,5.0,300.0,1.0,0.999331,0.001174,8
195,0.2,2.0,1.0,200.0,1.0,0.999290,0.001248,9
207,0.2,2.0,10.0,200.0,1.0,0.999289,0.001267,10


Jako optimální konfigurace Gradient Boostingu byla zvolena varianta s rychlostí učení learning_rate = 0.2, hloubkou slabých stromů max_depth = 2, minimální velikostí listu min_samples_leaf = 5, počtem stromů n_estimators = 300 a podílem trénovacích dat subsample = 0.8. Tato konfigurace dosáhla nejvyšší průměrné hodnoty ROC AUC v pětinásobné křížové validaci (0.999409 ± 0.000985), přičemž rozdíly oproti dalším konfiguracím byly menší než variabilita křížové validace, což potvrzuje stabilní a dobře generalizující chování modelu.